# SafeBank Assist — experiments E2, E3, E6 (Group 22)
PCAM ZG516 Applied Generative AI · `Qwen/Qwen2.5-1.5B-Instruct` on BANKING77

Runs top to bottom on a **free T4** (Runtime → Change runtime type → T4 GPU), after `safebank_colab.ipynb`'s fixes are on `main`. It is separate from that notebook so the two can be edited and re-run independently; the short setup is repeated here and the data is rebuilt deterministically (seed 42), so both notebooks use the same 385 evaluation rows.

| Experiment | What runs | Generations |
|---|---|---|
| **E2** | 4 prompt strategies × 385 queries, greedy decoding | 1,540 |
| **E3** | 6 decoding configurations × 77 queries (one per intent); sampled configurations once per seed (5 seeds), greedy once | 2,002 |
| **E6** | the six criteria on every condition, baseline vs final, at least 3 analysed failures | none (analysis only) |

**Speed.** Prompts carry all 77 intent labels (about 800 tokens; few-shot about 1,600), so one-at-a-time generation takes hours. `safebank.experiments` batch-generates every first attempt (batch size fixed at 16 and stored in each result) and replays them through the unchanged pipeline, so validation, repair, the confidence gate, tool routing and the safety screen all run as in production. Repair prompts, which are rare, are generated one at a time. `cache_hits` / `cache_misses` below show how much was batched.

**Evidence.** Everything written to `artifacts/results/` is report evidence. Commit it.

**Other CUDA machines.** It also runs on a Kubeflow JupyterLab server with a GPU: clone the repo in a terminal, open this notebook from the checkout, and run all. `experiments_device_report.json` records which GPU produced the numbers. If the server has no persistent volume, download the checkpoint zips as they appear.

## 1. Environment

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "No GPU: switch the runtime to T4"

name, memory.total [MiB], driver_version
NVIDIA A100-SXM4-80GB, 81920 MiB, 535.309.01


In [2]:
import os, subprocess

REPO_URL = "https://github.com/2025paml539-lab/safebank-assist-group22.git"
BRANCH = "main"
# Before the experiments PR is merged, main has no safebank.experiments. Either clone the
# fork branch in a terminal and open this notebook from that checkout (it is used as is),
# or swap in these two lines:
# REPO_URL = "https://github.com/rubberduckpanic/safebank-assist-group22.git"
# BRANCH = "2025paml561/s1-s2-report-and-experiments"


def find_checkout():
    """The repo root if this notebook is being run from inside a checkout."""
    path = os.getcwd()
    for _ in range(3):
        if os.path.isdir(os.path.join(path, "src", "safebank")):
            return path
        path = os.path.dirname(path)
    return None


root = find_checkout()
if root is None:
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL], check=True)
    root = os.path.abspath("safebank-assist-group22")
os.chdir(root)
print("repo:", root)
print(subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

repo: /home/jovyan/safebank-assist-group22
e7b1334 Notebook: run on a Kubeflow GPU server, save hardware report, checkpoint zips


In [3]:
# The model stack from requirements-colab.txt, plus the application layer in case the image lacks it
# (pip leaves anything already installed alone). torch is the image's own CUDA build.
!pip install -q -r requirements-colab.txt pydantic scikit-learn pandas

In [4]:
import sys, json
sys.path.insert(0, "src")
from safebank import config
from safebank.model_loader import device_report
for d in (config.RESULTS_DIR, config.FIGURES_DIR, config.DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)
# Saved with the results: the report must say which hardware produced the E2/E3 numbers.
hw = device_report().as_dict()
hw.update(seed=config.SEED, model_id=config.MODEL_ID)
(config.RESULTS_DIR / "experiments_device_report.json").write_text(json.dumps(hw, indent=2))
hw

{'device': 'cuda',
 'device_name': 'NVIDIA A100-SXM4-80GB',
 'cuda_capability': (8, 0),
 'vram_gb': 79.2,
 'torch_version': '2.8.0+cu128',
 'transformers_version': '5.19.0',
 'cuda_version': '12.8',
 'dtype': 'float16',
 'seed': 42,
 'model_id': 'Qwen/Qwen2.5-1.5B-Instruct'}

## 2. Data
Rebuilds the frozen 385-row evaluation sample (TEST split) and the few-shot bank (TRAIN split). `assert_no_overlap` stops the run if any few-shot example is also an evaluation query, which would contaminate every E2 number.

In [5]:
from safebank.data_prep import (
    load_banking77, dataset_label_space, build_eval_sample, build_example_bank,
)
from safebank.experiments import assert_no_overlap

train, test = load_banking77()
LABELS = dataset_label_space(train)
eval_rows = build_eval_sample(test, LABELS)            # 5 x 77 = 385, seed 42, TEST split
EXAMPLES = build_example_bank(                         # TRAIN split, never an eval query
    train, LABELS, exclude={r["query"] for r in eval_rows}
)
assert_no_overlap(EXAMPLES, eval_rows)
print(len(LABELS), "intents;", len(eval_rows), "eval rows;", len(EXAMPLES), "few-shot examples")

77 intents; 385 eval rows; 4 few-shot examples


## 3. Model
Qwen2.5-1.5B-Instruct in fp16. `HFGenerator` resets Qwen's shipped sampling defaults to neutral, so each decoding configuration in E3 means what its label says.

In [6]:
from safebank.model_loader import load_model_and_tokenizer, HFGenerator

model, tokenizer = load_model_and_tokenizer()
GEN = HFGenerator(model, tokenizer)
BATCH_SIZE = 16        # keep fixed across E2 and E3; stored in every result
print(config.MODEL_ID, "| context window", GEN.context_window, "| batch size", BATCH_SIZE)

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen/Qwen2.5-1.5B-Instruct | context window 32768 | batch size 16


## 4. E2 — prompt strategies
Same pipeline, same 385 queries, greedy decoding; only the prompt strategy changes. Few-shot receives the real train-split bank. Qwen's chat template adds a default system prompt to every strategy (report §7), so comparisons stay fair.

In [7]:
from safebank.experiments import run_e2, report_table, save_condition, save_json
from safebank.evaluate import results_dataframe

e2 = run_e2(GEN, LABELS, eval_rows, EXAMPLES, batch_size=BATCH_SIZE)
for cond in e2:
    save_condition(cond, f"e2_{cond.strategy.value}")
save_json([c.summary() for c in e2], "e2_summary")
import shutil
print("checkpoint:", shutil.make_archive("safebank_e2_checkpoint", "zip", str(config.RESULTS_DIR)))
results_dataframe(report_table(e2))

[transformers] The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


checkpoint: /home/jovyan/safebank-assist-group22/safebank_e2_checkpoint.zip


,strategy,accuracy,macro_f1,schema_first_attempt_rate,schema_final_rate,tool_routing_accuracy,escalation_rate,requires_human_rate,safety_unsafe_rate,mean_attempts,mean_latency_ms
0,zero_shot,0.4026,0.3621,0.9506,0.9532,0.62,0.2805,0.0260,0.1740,1.096,402.0
1,few_shot,0.4779,0.4501,0.8338,0.9481,0.68,0.2104,0.1091,0.0390,1.221,459.2
2,role_context,0.3896,0.3465,0.9506,0.9584,0.68,0.3039,0.0597,0.1896,1.091,426.5
3,constraint_based,0.4000,0.3677,0.9481,0.9481,0.56,0.2390,0.0286,0.1299,1.104,359.5


In [8]:
# How much was batched, and what it cost (criterion 5).
for cond in e2:
    m = cond.meta
    print(f"{cond.strategy.value:17s} hits={m['cache_hits']:4d} misses={m['cache_misses']:3d} wall={m['wall_seconds']:7.1f}s")
u = GEN.usage_log
print(f"\n{len(u)} generations | mean prompt tokens {sum(x.prompt_tokens for x in u)/len(u):.0f} | "
      f"mean completion tokens {sum(x.completion_tokens for x in u)/len(u):.0f} | truncated prompts {sum(x.truncated for x in u)}")

zero_shot         hits= 385 misses= 37 wall=  154.8s
few_shot          hits= 385 misses= 85 wall=  176.8s
role_context      hits= 385 misses= 35 wall=  164.2s
constraint_based  hits= 385 misses= 40 wall=  138.4s

1737 generations | mean prompt tokens 845 | mean completion tokens 79 | truncated prompts 0


## 5. E3 — decoding grid
Runs on the best E2 strategy, over a stratified 77-row subset (one query per intent). Each sampled configuration runs once per seed; **greedy runs once** because it is deterministic, so it has no stability score. `mean_agreement` is the share of seeds that match each query's most common prediction (1.0 = every seed agreed).

In [9]:
from safebank.experiments import (
    best_condition, select_e3_subset, strategy_kwargs, run_e3, save_grid,
    sample_outputs, variability_examples,
)

BEST = best_condition(e2)
print("E3 runs on the best E2 strategy:", BEST.strategy.value)
subset = select_e3_subset(eval_rows)                   # 1 query per intent = 77 rows
e3 = run_e3(
    GEN, LABELS, subset, BEST.strategy,
    prompt_kwargs=strategy_kwargs(BEST.strategy, EXAMPLES), batch_size=BATCH_SIZE,
)
save_grid(e3)
save_json([g.summary() for g in e3], "e3_summary")
import shutil
print("checkpoint:", shutil.make_archive("safebank_e3_checkpoint", "zip", str(config.RESULTS_DIR)))
results_dataframe([g.summary() for g in e3])

E3 runs on the best E2 strategy: few_shot
checkpoint: /home/jovyan/safebank-assist-group22/safebank_e3_checkpoint.zip


,decoding,n_runs,accuracy_mean,accuracy_min,accuracy_max,schema_first_attempt_rate,schema_final_rate,escalation_rate,mean_attempts,mean_agreement,fully_stable_fraction
0,greedy,1,0.4805,0.4805,0.4805,0.8182,0.9351,0.2078,1.2600,NaN,NaN
1,temperature=0.7,5,0.4467,0.4286,0.4545,0.9013,0.9480,0.2701,1.1508,0.8519,0.5844
2,temperature=1.2,5,0.3948,0.3636,0.4416,0.7922,0.8442,0.3740,1.3666,0.7195,0.2597
3,top_k=50 temp=0.7,5,0.4467,0.4286,0.4545,0.9039,0.9480,0.2701,1.1482,0.8519,0.5844
4,top_p=0.9 temp=0.7,5,0.4494,0.4286,0.4675,0.8857,0.9455,0.2805,1.1716,0.8649,0.6104
5,top_p=0.95 temp=0.7,5,0.4493,0.4156,0.4805,0.8779,0.9480,0.2779,1.1742,0.8623,0.5974


In [10]:
# Three sample outputs per configuration, and the queries where seeds disagree.
samples = sample_outputs(e3, k=3)
variability = {g.point.label: variability_examples(g, k=3) for g in e3}
save_json({"samples": samples, "variability": variability}, "e3_samples")
for label, replies in samples.items():
    print("\n==", label)
    for s in replies:
        print("  -", s["query"][:60], "->", s["intent"], "|", (s["response"] or "")[:60])
for label, examples in variability.items():
    if examples:
        print("\nSeeds disagree under", label)
        for ex in examples:
            print("  -", ex["query"][:60], "| gold", ex["gold_intent"], "|", ex["predictions_by_seed"])


== greedy
  - I need to activate my card can you do that now? -> activate_my_card | I can help you activate your card now. Please provide your c
  - How do I go about setting up an account for my daughter? -> get_disposable_virtual_card | I can help you with that. Let me guide you through the proce
  - Can I use the system Google Pay for top-ups? -> apple_pay_or_google_pay | Yes, you can use Google Pay for top-ups. Would you like to p

== temperature=0.7
  - I need to activate my card can you do that now? -> activate_my_card | I can help with that. Please follow the prompts to activate 
  - How do I go about setting up an account for my daughter? -> get_disposable_virtual_card | I can guide you through setting up a disposable virtual card
  - Can I use the system Google Pay for top-ups? -> apple_pay_or_google_pay | I can help you with that. Google Pay can be used for top-ups

== temperature=1.2
  - I need to activate my card can you do that now? -> activate_my_card | Sure, I can guide

## 6. E6 — the six criteria, baseline vs final, failure analysis
The six criteria are in every row above: accuracy and macro-F1, schema validity before and after repair, tool routing, reply safety, latency and attempts, and stability across seeds (E3). Baseline is the zero-shot prompt; final is the best E2 strategy. If they are the same strategy, the comparison is empty by construction: say so in the report.

Failure cases are drawn across different classes (each validation failure kind, a predicted confusable pair, within-theme, cross-theme) so the root-cause table is not three copies of one problem.

In [11]:
from safebank.experiments import compare_conditions, failure_cases, safety_findings

BASELINE = next(c for c in e2 if c.strategy.value == "zero_shot")
comparison = compare_conditions(BASELINE, BEST)
cases = failure_cases(BEST, n=6)
unsafe = safety_findings(BEST)
save_json(
    {"baseline": BASELINE.strategy.value, "final": BEST.strategy.value,
     "comparison": comparison, "failure_cases": cases, "safety_findings": unsafe},
    "e6_analysis",
)
print(f"{len(cases)} failure cases | {len(unsafe)} replies flagged by the safety screen")
results_dataframe(comparison)

6 failure cases | 15 replies flagged by the safety screen


,metric,baseline,final,change
0,accuracy,0.4026,0.4779,0.0753
1,macro_f1,0.3621,0.4501,0.0880
2,schema_first_attempt_rate,0.9506,0.8338,-0.1168
3,schema_final_rate,0.9532,0.9481,-0.0051
4,tool_routing_accuracy,0.6200,0.6800,0.0600
5,escalation_rate,0.2805,0.2104,-0.0701
6,safety_unsafe_rate,0.1740,0.0390,-0.1350
7,mean_attempts,1.0960,1.2210,0.1250
8,mean_latency_ms,402.0000,459.2000,57.2000


In [12]:
if cases:
    display(results_dataframe(cases)[["category", "gold_intent", "predicted_intent", "query", "note"]])
unsafe

,category,gold_intent,predicted_intent,query,note
0,confusable_pair,card_arrival,card_delivery_estimate,Do you know if there is a tracking number for ...,Predicted confusable pair: 'card_arrival' vs '...
1,cross_theme,age_limit,get_disposable_virtual_card,How do I go about setting up an account for my...,Cross-theme misclassification: 'age_limit' pre...
2,validation:schema_violation,apple_pay_or_google_pay,NaN,"I got my American Express in Apple Pay, why is...",Parsed but violated the pydantic contract - a ...
3,validation:unknown_intent,disposable_card_limits,NaN,How can i get multiple disposble cards.,Schema-valid but the label is not one of the 7...
4,within_theme,failed_transfer,pending_transfer,Can you help me figure out what's happening? I...,Within-theme confusion in 'transfer': 'failed_...
5,confusable_pair,card_delivery_estimate,card_arrival,How long should I expect for the card to arrive?,Predicted confusable pair: 'card_delivery_esti...


[{'query': "Please put a freeze on my card,  I am worried there has been some payments on it and I don't know what for.",
  'gold_intent': 'card_payment_not_recognised',
  'reply': 'I understand your concern. I will put a freeze on your card to prevent any unauthorized transactions. Please provide your account number and PIN so I can proceed with the freeze.',
  'issues': 'credential_solicitation',
  'withheld': True},
 {'query': 'How might  I change my PIN?',
  'gold_intent': 'change_pin',
  'reply': "I can help you with that. Please provide your account number and the new PIN you'd like to set.",
  'issues': 'credential_solicitation',
  'withheld': True},
 {'query': 'I want to set a new PIN',
  'gold_intent': 'change_pin',
  'reply': "Sure, I can help you with that. Please provide your current PIN and the new PIN you'd like to set.",
  'issues': 'credential_solicitation',
  'withheld': True},
 {'query': 'I need to change my pin but I am on the road.',
  'gold_intent': 'change_pin',
 

## 7. Where each result goes in the report

| Result | File in `artifacts/results/` | Report |
|---|---|---|
| E2 comparison table | `e2_summary.json`, `e2_<strategy>.json` | §7, §9 sentence, §1 and §14 headline |
| E4 validity before / after repair | `schema_*` fields in `e2_summary.json` | §10.1, §11 |
| E3 table, samples, stability | `e3_summary.json`, `e3_samples.json`, `e3_<config>_seed<n>.json` | §8 |
| E6 comparison, failures, safety | `e6_analysis.json` | §11, §12.1 |
| Validated, repaired and escalated replies | `attempts_raw` and `escalated` in `e2_<strategy>.json` | Appendix E |

## 8. Save the evidence
Zips `artifacts/results/` for download. Put the contents back into the repo's `artifacts/results/` on the branch and commit them: the report sections are written from these files.

In [13]:
import shutil
archive = shutil.make_archive("safebank_experiment_results", "zip", str(config.RESULTS_DIR))
print(archive, "|", len(list(config.RESULTS_DIR.glob("*.json"))), "json files")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Not on Colab: the zip is in the working directory.")


/home/jovyan/safebank-assist-group22/safebank_experiment_results.zip | 38 json files
Not on Colab: the zip is in the working directory.
